# Strike Extraction

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tritonoa.data.time import TIME_PRECISION

from vineyard.config import get_path
from vineyard.readers import read_acoustic_data
from vineyard.signal import find_strikes

In [ ]:
sensors = [
    {"name": "3dvha", "channel": 7, "distance_sec": 1.0, "threshold": 0.05},
    {"name": "vla1", "channel": 3, "distance_sec": 1.0, "threshold": 0.05},
    {"name": "vla2", "channel": 0, "distance_sec": 1.0, "threshold": 0.02},
]
time_start = np.datetime64("2023-12-01T21:51:00.00", TIME_PRECISION)
time_end = np.datetime64("2023-12-01T22:26:00.00", TIME_PRECISION)


fig, axs = plt.subplots(nrows=3, figsize=(10, 8), sharex=True)

for ax, sensor in zip(axs, sensors):
    name, channel, distance_sec, threshold = tuple(sensor.values())
    ds = read_acoustic_data(
        get_path(f"{name}_inventory"),
        time_start,
        time_end,
        channels=channel,
        taper_pc=1e-4,
        dec_factor=None,
        filt_type="bandpass",
        filt_freq=[100.0, 300.0],
    )
    peaks = find_strikes(ds.data[0], ds.stats.sampling_rate, threshold, distance_sec)

    cf = ds.data[0] ** 2 / np.max(ds.data[0] ** 2)

    ax.plot(ds.time_vector, cf, label="Signal")
    ax.plot(ds.time_vector[peaks], cf[peaks], "ro", label="Peaks")
    ax.axhline(threshold, color="red", linestyle="--", label="Threshold")
    ax.set_title(f"{name} - Num detections: {len(peaks)}\nFirst detection: {ds.time_vector[peaks[0]]}")

    plt.tight_layout()
plt.show()